# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm.notebook import tqdm
import itertools

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

1. Прочитайте файл [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). Он похож на тот, что был в предыдущем упражнении, но на этот раз мы не масштабировали непрерывные объекты (мы больше не собираемся использовать logreg). Не забудьте дополнить таблицу столбцом "день недели" из csv-файла за предыдущий день.
2. Используя `train_test_split` с параметрами `test_size=0.2`, `random_state=21`, получаем `X_train`, `y_train`, `X_test`, `y_test". Используйте дополнительный параметр `стратифицировать`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')

df_target = pd.read_csv('../data/dayofweek.csv')

In [3]:
if len(df) == len(df_target):
    df['dayofweek'] = df_target['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=21, 
    stratify=y
)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

1. Используя `GridSearchCV`, попробуйте различные параметры ядра (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), гамма (`scale`, `auto`), class_weight (`сбалансированный`, `Нет`) используют `random_state=21` и `probability=True` и получают наилучшую комбинацию из них с точки зрения точности.
2. Создайте фрейм данных на основе результатов сетевого поиска и отсортируйте его по возрастанию в соответствии с "rank_test_score". Проверьте, есть ли большая разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).

In [4]:
svc = SVC(random_state=21, probability=True)
param_grid_svc = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None]
}

grid_svc = GridSearchCV(estimator=svc, param_grid=param_grid_svc, scoring='accuracy', n_jobs=-1)
grid_svc.fit(X_train, y_train)

print(f"Best parameters: {grid_svc.best_params_}")
print(f"Best score: {grid_svc.best_score_:.5f}")

Best parameters: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Best score: 0.87611


In [5]:
df_svc_results = pd.DataFrame(grid_svc.cv_results_)
df_svc_results = df_svc_results.sort_values(by='rank_test_score')
df_svc_results[['params', 'mean_test_score', 'rank_test_score']].head()

,params,mean_test_score,rank_test_score
70,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.876109,1
64,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.863500,2
58,"{'C': 5, 'class_weight': None, 'gamma': 'auto'...",0.816018,3
52,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.807865,4
63,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.721052,5


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

1. Используя "GridSearchCV", попробуйте использовать различные параметры "max_depth" (от "1" до "49"), "class_weight" ("сбалансированный", "Нет") и "критерий" ("энтропия" и "джини") и получите наилучшую комбинацию из них с точки зрения точности. Используйте `random_state=21`.
2. Создайте фрейм данных на основе результатов сетевого поиска и отсортируйте его по возрастанию по "rank_test_score", проверьте, есть ли большая разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).

In [6]:
dt = DecisionTreeClassifier(random_state=21)
param_grid_dt = {
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

grid_dt = GridSearchCV(estimator=dt, param_grid=param_grid_dt, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_train, y_train)

print(f"Best parameters: {grid_dt.best_params_}")
print(f"Best score: {grid_dt.best_score_:.5f}")

Best parameters: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': np.int64(23)}
Best score: 0.87386


In [7]:
df_dt_results = pd.DataFrame(grid_dt.cv_results_)
df_dt_results = df_dt_results.sort_values(by='rank_test_score')
df_dt_results[['params', 'mean_test_score', 'rank_test_score']].head()

,params,mean_test_score,rank_test_score
97,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
95,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
94,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
71,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1
72,"{'class_weight': 'balanced', 'criterion': 'gin...",0.873859,1


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

1. Используя "GridSearchCV", попробуйте использовать другие параметры "n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (от `1` до `49`), `class_weight` ("сбалансированный", "Нет") и "критерий` (`энтропия` и `джини`) и получить наилучшую комбинацию из них с точки зрения точности. Используйте random_state=21.
2. Создайте фрейм данных на основе результатов сетевого поиска и отсортируйте его по возрастанию с помощью `rank_test_score`, проверьте, есть ли большая разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).

In [8]:
rf = RandomForestClassifier(random_state=21)
param_grid_rf = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini']
}

grid_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"Best parameters: {grid_rf.best_params_}")
print(f"Best score: {grid_rf.best_score_:.5f}")

Best parameters: {'class_weight': None, 'criterion': 'gini', 'max_depth': np.int64(28), 'n_estimators': 50}
Best score: 0.90429


In [9]:
df_rf_results = pd.DataFrame(grid_rf.cv_results_)
df_rf_results = df_rf_results.sort_values(by='rank_test_score')
df_rf_results[['params', 'mean_test_score', 'rank_test_score']].head()

,params,mean_test_score,rank_test_score
698,"{'class_weight': None, 'criterion': 'gini', 'm...",0.904290,1
711,"{'class_weight': None, 'criterion': 'gini', 'm...",0.904287,2
374,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
390,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3
386,"{'class_weight': 'balanced', 'criterion': 'gin...",0.903549,3


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

Поиск по сетке может быть довольно долгим процессом, и вы можете задаться вопросом, когда же он закончится.
1. Создайте ручной поиск по сетке для тех же значений параметров случайного леса, повторяя список возможных значений и вычисляя "cross_val_score" для каждой комбинации. Попробуйте увеличить "n_jobs". Значение "cv" для "cross_val_score" равно 5.
2. Отслеживайте прогресс, используя библиотеку `tqdm.notebook".
3. Создайте фрейм данных из результатов сетевого поиска со столбцами, соответствующими названиям параметров, а также `mean_accuracy` и `std_accuracy`.
4. Отсортируйте его по убыванию с помощью `mean_accuracy`, проверьте, есть ли большая разница между различными комбинациями (иногда более простая модель может дать сопоставимый результат).

In [10]:
n_estimators_list = [5, 10, 50, 100]
max_depth_list = np.arange(1, 50)
class_weight_list = ['balanced', None]
criterion_list = ['entropy', 'gini']

param_combinations = list(itertools.product(
    n_estimators_list, max_depth_list, class_weight_list, criterion_list
))

results = []
for params in tqdm(param_combinations, desc="GridSearch Progress"):
    n_est, depth, weight, crit = params
    
    model = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=depth,
        class_weight=weight,
        criterion=crit,
        random_state=21,
        n_jobs=-1
    )
    
    scores = cross_val_score(model, X_train, y_train, cv=5)
    
    results.append({
        'n_estimators': n_est,
        'max_depth': depth,
        'class_weight': weight,
        'criterion': crit,
        'mean_accuracy': scores.mean(),
        'std_accuracy': scores.std()
    })

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
df_manual_results = pd.DataFrame(results)
df_manual_results = df_manual_results.sort_values(by='mean_accuracy', ascending=False)
df_manual_results.head()

,n_estimators,max_depth,class_weight,criterion,mean_accuracy,std_accuracy
503,50,28,None,gini,0.904290,0.010961
711,100,31,None,gini,0.903547,0.014380
509,50,30,balanced,gini,0.902817,0.013554
525,50,34,balanced,gini,0.902809,0.013010
783,100,49,None,gini,0.902806,0.010460


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred = best_rf.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)
print(f"Final Accuracy on Test Set: {final_acc:.5f}")

Final Accuracy on Test Set: 0.92899
